<img src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/brewly_logo.png"/>

# Brewly database to AWS RDS PostgreSQL ☕️

This notebook is the final step of the Brewly pipeline. The local SQLite file `brewly.db` was good for practice. It cannot be shared with the team. It cannot be queried from a dashboard. It cannot survive a laptop crash. A managed PostgreSQL database on AWS RDS solves all three problems.

<Note type="important" title="Just follow the steps!">

In this exercise, you will follow the steps to create a PostgreSQL database on AWS RDS, connect to it from your local machine, and load the data from `brewly.db` into it. You will not write any SQL queries in this notebook. You will only use Python to run the commands that set up the database and move the data. Just follow the steps.
No solutions are provided for this notebook. If you get stuck, ask for help to your classmates or to your TA.

The goal of this exercise is to get familiar with the process of setting up a database and loading data into it. You will do this many times in your career, so it's important to understand the steps and the tools involved.
</Note>

## Step 0: Create the RDS instance in the AWS Console

This step is done once, in the browser. You do not run code for it.

Log in to the [AWS Console](https://signin.aws.amazon.com/signin?client_id=arn%3Aaws%3Asignin%3A%3A%3Aconsole%2Fcanvas&redirect_uri=https%3A%2F%2Fconsole.aws.amazon.com%2Fconsole%2Fhome%3Fca-oauth-flow-id%3D15ed%26hashArgs%3D%2523%26isauthcode%3Dtrue%26oauthStart%3D1778580289843%26state%3DhashArgsFromTB_eu-north-1_fb99b42e38f7d02c&page=resolve&code_challenge=-d-vsBNN6Cz8oljX1lMuCrPB69Y0uy3LoLlvq5fLUYk&code_challenge_method=SHA-256&backwards_compatible=true). Open the **RDS** service. Click **Create database**.

<Video video ="https://vimeo.com/1191509754?share=copy&fl=sv&fe=ci" />

Choose the following options:

- Engine: **PostgreSQL**
- Template: **Free tier**
- DB instance identifier: `brewly-db`
- Master username: `postgres`
- Master password: pick a strong password and save it
- Instance class: `db.t4g.micro` (free tier)
- Storage: 20 GB, default settings
- Public access: **Yes** (so you can connect from this notebook)
- VPC security group: create new, name it `brewly-sg`
- Initial database name: `brewly`

Click **Create database**. The instance takes about 5 minutes to become available.

<img src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/rds_create_db.png"/>

Once the status shows **Available**, open the instance and copy the **Endpoint** value. It looks like `brewly-db.abc123xyz.eu-west-3.rds.amazonaws.com`. You need this string to connect.

Finally, open the security group `brewly-sg`. Edit the inbound rules. Add a rule that allows PostgreSQL (port 5432) from your IP address. Without this rule the connection will time out.

<Note type="important" title="Security note">
opening port 5432 to your IP is fine for practice. In production, the database lives in a private subnet and only application servers can reach it. Never open RDS to the public internet.
</Note>

## Setup

Install `psycopg2` and import the libraries you need. `psycopg2` is the standard Python driver for PostgreSQL.

In [1]:
%pip install psycopg2-binary pandas --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


psycopg2 is the equivalent of `sqlite3` for PostgreSQL. It has the same basic workflow: connect, create a cursor, execute queries, fetch results, close the connection.

In [2]:
import sqlite3
import psycopg2
import pandas as pd

## Step 1: Read the data from the local SQLite file

The previous notebook saved the data to `brewly.db`. Open that file, read both tables into DataFrames, and confirm the row counts.

In [3]:
# Open the local SQLite file from the previous exercise
sqlite_conn = sqlite3.connect("brewly.db")

# Read both tables into DataFrames
customers_df = pd.read_sql_query("SELECT * FROM customers;", sqlite_conn)
orders_df = pd.read_sql_query("SELECT * FROM orders;", sqlite_conn)

# Close the SQLite connection: we have the data in memory now
sqlite_conn.close()

print("Customers rows:", len(customers_df))
print("Orders rows:", len(orders_df))
print()
print("Customers preview:")
print(customers_df.head(3))

Customers rows: 8
Orders rows: 11

Customers preview:
   customer_id           name  country signup_date
0            1  Marie Laurent   France  2024-01-15
1            2  Tom Schneider  Germany  2024-02-03
2            3    Sofia Rossi    Italy  2024-02-20


## Step 2: Connect to the RDS PostgreSQL instance

Replace the placeholder values below with your real RDS endpoint and password. Then open the connection and enable autocommit so each statement runs immediately.

In [4]:
# Connection settings: replace HOST and PASSWORD with your real values
import os
from dotenv import load_dotenv
load_dotenv()

RDS_HOST = os.getenv("RDS_HOST")
RDS_PORT = int(os.getenv("RDS_PORT"))
RDS_DBNAME = os.getenv("RDS_DBNAME")
RDS_USER = os.getenv("RDS_USER")
RDS_PASSWORD = os.getenv("RDS_PASSWORD")

# Open the connection
pg_conn = psycopg2.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    dbname=RDS_DBNAME,
    user=RDS_USER,
    password=RDS_PASSWORD,
)

# Autocommit means every statement is saved right away
# This is fine for setup. For batch inserts, you can turn it off and commit at the end.
pg_conn.autocommit = True

# Create a cursor to run SQL commands
pg_cursor = pg_conn.cursor()

# Check the connection works by asking PostgreSQL for its version
pg_cursor.execute("SELECT version();")
version_row = pg_cursor.fetchone()
print("Connected to:", version_row[0])

Connected to: PostgreSQL 18.3 on aarch64-unknown-linux-gnu, compiled by aarch64-unknown-linux-gnu-gcc (GCC) 12.4.0, 64-bit


## Step 3: Create the two tables on RDS

SQLite is forgiving with types. PostgreSQL is strict. You must declare every column with the right type. Drop the tables first so the script is repeatable.

In [7]:
# Drop the tables if they already exist, so the script can run again cleanly
pg_cursor.execute("DROP TABLE IF EXISTS orders;")
pg_cursor.execute("DROP TABLE IF EXISTS customers;")

# Create the customers table
# customer_id is the primary key: it identifies each customer uniquely
create_customers_sql = """
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    country     TEXT NOT NULL,
    signup_date DATE NOT NULL
);
"""
pg_cursor.execute(create_customers_sql)

# Create the orders table
# customer_id is a foreign key: it points at customers.customer_id
create_orders_sql = """
CREATE TABLE orders (
    order_id     INTEGER PRIMARY KEY,
    customer_id  INTEGER NOT NULL REFERENCES customers(customer_id),
    order_date   DATE NOT NULL,
    amount_eur   NUMERIC(10, 2) NOT NULL,
    product      TEXT NOT NULL
);
"""
pg_cursor.execute(create_orders_sql)

print("Tables created on RDS.")

Tables created on RDS.


## Step 4: Insert the customers

Loop through each row of the DataFrame and insert it. Use `%s` placeholders so `psycopg2` handles quoting and escaping safely. Never build SQL with f-strings or string concatenation: that opens the door to SQL injection.

In [ ]:
# The INSERT statement uses %s placeholders for each column
insert_customer_sql = """
INSERT INTO customers (customer_id, name, country, signup_date)
VALUES (%s, %s, %s, %s);
"""

# Loop through the DataFrame, one row at a time
inserted_customers = 0
for index, row in customers_df.iterrows():
    customer_id = int(row["customer_id"])
    name = row["name"]
    country = row["country"]
    signup_date = row["signup_date"]

    # Build the tuple of values that match the %s placeholders
    values = (customer_id, name, country, signup_date)
    pg_cursor.execute(insert_customer_sql, values)
    inserted_customers = inserted_customers + 1

print("Customers inserted:", inserted_customers)


UniqueViolation: duplicate key value violates unique constraint "customers_pkey"
DETAIL:  Key (customer_id)=(1) already exists.


## Step 5: Insert the orders

Same pattern as the customers. The foreign key on `customer_id` means every order must reference an existing customer. If a `customer_id` does not exist in the `customers` table, PostgreSQL refuses the insert.

In [10]:
insert_order_sql = """
INSERT INTO orders (order_id, customer_id, order_date, amount_eur, product)
VALUES (%s, %s, %s, %s, %s);
"""

inserted_orders = 0
for index, row in orders_df.iterrows():
    order_id = int(row["order_id"])
    customer_id = int(row["customer_id"])
    order_date = row["order_date"]
    amount_eur = float(row["amount_eur"])
    product = row["product"]
    print(index)

    values = (order_id, customer_id, order_date, amount_eur, product)
    pg_cursor.execute(insert_order_sql, values)
    inserted_orders = inserted_orders + 1

print("Orders inserted:", inserted_orders)

0
1
2
3
4
5
6
7
8
9
10
Orders inserted: 11


## Step 6: Verify the data on RDS

Run a few `SELECT` queries to confirm the data is actually on the server. The numbers must match the row counts from Step 1.

In [11]:
# Count rows on the RDS side
pg_cursor.execute("SELECT COUNT(*) FROM customers;")
customers_count = pg_cursor.fetchone()
print("Customers on RDS:", customers_count[0])

pg_cursor.execute("SELECT COUNT(*) FROM orders;")
orders_count = pg_cursor.fetchone()
print("Orders on RDS:", orders_count[0])

Customers on RDS: 8
Orders on RDS: 11


In [12]:
# Re-run the top-customers query from the previous notebook, this time on RDS
top_customers_sql = """
SELECT c.name, SUM(o.amount_eur) AS total_spent
FROM customers AS c
INNER JOIN orders AS o ON c.customer_id = o.customer_id
GROUP BY c.name
ORDER BY total_spent DESC
LIMIT 3;
"""

pg_cursor.execute(top_customers_sql)
top_customers = pg_cursor.fetchall()

print("Top three customers (from RDS):")
for row in top_customers:
    print(row)

Top three customers (from RDS):
('Pierre Dubois', Decimal('188.00'))
('Marie Laurent', Decimal('140.00'))
('Tom Schneider', Decimal('136.00'))


## Step 7: Read RDS results back into a DataFrame

`pd.read_sql_query()` works with any database connection, not just SQLite. Pass the PostgreSQL connection and the same query.

In [13]:
# Same query, but the result lands directly in a DataFrame
top_customers_df = pd.read_sql_query(top_customers_sql, pg_conn)
print(top_customers_df)

            name  total_spent
0  Pierre Dubois        188.0
1  Marie Laurent        140.0
2  Tom Schneider        136.0


/var/folders/z5/kyyhqxqd4b966377jmszys600000gn/T/ipykernel_21801/2922468569.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  top_customers_df = pd.read_sql_query(top_customers_sql, pg_conn)


## Step 8: Close the connection

Always close the cursor and the connection at the end. RDS limits the number of open connections per instance. Leaking connections will eventually block new logins.

In [14]:
pg_cursor.close()
pg_conn.close()
print("Cursor and connection closed.")

Cursor and connection closed.
